In [1]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

In [2]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)
pd.options.display.float_format = '{:.2f}'.format

1. Collecting raw data

The six main factors, effecting the risk of an asthma excerbation, investigated in this prototype are: Particulate Matter <= 2.5 micrograms (PM2.5), nitrogen dioxide (NO2), ozone (O3), pollen, relative Humidity and temperature. Raw data is collected from Open Meteo.

In [3]:
# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://air-quality-api.open-meteo.com/v1/air-quality"
params = {
	"latitude": 34.80,
	"longitude": 38.99,
	"hourly": ["pm2_5", "nitrogen_dioxide", "ozone",],
    "timezone": "auto",
	"past_days": 1,
}
air_quality_responses = openmeteo.weather_api(url, params = params)


Defining a python list with of the hourly and current raw data values.

In [4]:
# Process first location. Add a for-loop for multiple locations or weather models
airQualityResponse = air_quality_responses[0]
# Process hourly data. The order of variables needs to be the same as requested.
hourly_AQ = airQualityResponse.Hourly()
hourly_pm2_5 = hourly_AQ.Variables(0).ValuesAsNumpy()
hourly_nitrogen_dioxide = hourly_AQ.Variables(1).ValuesAsNumpy()
hourly_ozone = hourly_AQ.Variables(2).ValuesAsNumpy()

hourly_air_quality_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly_AQ.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly_AQ.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly_AQ.Interval()),
		inclusive = "left"
	).tz_convert(airQualityResponse.Timezone().decode())
}

hourly_air_quality_data["pm2_5"] = hourly_pm2_5
hourly_air_quality_data["nitrogen_dioxide"] = hourly_nitrogen_dioxide
hourly_air_quality_data["ozone"] = hourly_ozone

hourly_air_quality_dataframe = pd.DataFrame(data = hourly_air_quality_data)
print("\n This is the Hourly Air Quality Index (AQI) DataFrame:")
hourly_air_quality_dataframe.head(72)


 This is the Hourly Air Quality Index (AQI) DataFrame:


,date,pm2_5,nitrogen_dioxide,ozone
0,2026-09-05 00:00:00+03:00,20.60,0.50,89.00
1,2026-09-05 01:00:00+03:00,19.00,0.50,85.00
2,2026-09-05 02:00:00+03:00,18.00,0.50,85.00
3,2026-09-05 03:00:00+03:00,16.90,0.50,86.00
4,2026-09-05 04:00:00+03:00,15.30,0.50,87.00
...,...,...,...,...
67,2026-09-07 19:00:00+03:00,7.10,0.60,91.00
68,2026-09-07 20:00:00+03:00,5.90,0.70,86.00
69,2026-09-07 21:00:00+03:00,5.40,0.80,87.00
70,2026-09-07 22:00:00+03:00,5.80,0.80,87.00


In [5]:
url = "https://air-quality-api.open-meteo.com/v1/air-quality"
params = {
	"latitude": 34.80,
	"longitude": 38.99,
	"hourly": ["birch_pollen", "grass_pollen", "ragweed_pollen"],
	"timezone": "auto",
	"past_days": 4,
}
pollen_responses = openmeteo.weather_api(url, params = params)

In [6]:
# Process first location. Add a for-loop for multiple locations or weather models
pollenResponse = pollen_responses[0]

# Process hourly data. The order of variables needs to be the same as requested.
hourlyPollen = pollenResponse.Hourly()
hourly_birch_pollen = hourlyPollen.Variables(0).ValuesAsNumpy()
hourly_grass_pollen = hourlyPollen.Variables(1).ValuesAsNumpy()
hourly_ragweed_pollen = hourlyPollen.Variables(2).ValuesAsNumpy()

hourly_pollen_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourlyPollen.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourlyPollen.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourlyPollen.Interval()),
		inclusive = "left"
	).tz_convert(pollenResponse.Timezone().decode())
}

hourly_pollen_data["birch_pollen"] = hourly_birch_pollen
hourly_pollen_data["grass_pollen"] = hourly_grass_pollen
hourly_pollen_data["ragweed_pollen"] = hourly_ragweed_pollen

hourly_pollen_df = pd.DataFrame(data = hourly_pollen_data)
hourly_pollen_df.head(20)

,date,birch_pollen,grass_pollen,ragweed_pollen
0,2026-09-02 00:00:00+03:00,0.00,0.40,0.30
1,2026-09-02 01:00:00+03:00,0.00,0.40,0.30
2,2026-09-02 02:00:00+03:00,0.00,0.30,0.30
3,2026-09-02 03:00:00+03:00,0.00,0.30,0.20
4,2026-09-02 04:00:00+03:00,0.00,0.20,0.20
5,2026-09-02 05:00:00+03:00,0.00,0.20,0.20
6,2026-09-02 06:00:00+03:00,0.00,0.20,0.10
7,2026-09-02 07:00:00+03:00,0.00,0.20,0.10
8,2026-09-02 08:00:00+03:00,0.00,0.20,0.10
9,2026-09-02 09:00:00+03:00,0.00,0.30,0.10


In [7]:
# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 34.80,
	"longitude": 38.99,
	"daily": ["temperature_2m_max", "temperature_2m_min"],
	"hourly": "relative_humidity_2m",
	"timezone": "auto",
	"past_days": 2,
}
weather_responses = openmeteo.weather_api(url, params = params)

Spezifizierung von weather_responses als Liste für Max und Min Temperaturen sowie relative Luftfeuchtigkeit.

In [8]:
weatherResponse = weather_responses[0]

# Process hourly data. The order of variables needs to be the same as requested.
hourlyRH = weatherResponse.Hourly()
hourly_relative_humidity_2m = hourlyRH.Variables(0).ValuesAsNumpy()

hourly_RH_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourlyRH.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourlyRH.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourlyRH.Interval()),
		inclusive = "left"
	).tz_convert(weatherResponse.Timezone().decode())
}

hourly_RH_data["relative_humidity_2m"] = hourly_relative_humidity_2m

hourly_RH_dataframe = pd.DataFrame(data = hourly_RH_data)
print("\n This is the Hourly Relative Humidity DataFrame:")
hourly_RH_dataframe.head(5)


 This is the Hourly Relative Humidity DataFrame:


,date,relative_humidity_2m
0,2026-09-04 00:00:00+03:00,23.00
1,2026-09-04 01:00:00+03:00,25.00
2,2026-09-04 02:00:00+03:00,27.00
3,2026-09-04 03:00:00+03:00,28.00
4,2026-09-04 04:00:00+03:00,28.00


In [9]:

# Process daily data. The order of variables needs to be the same as requested.
dailyTemp = weatherResponse.Daily()
daily_temperature_2m_max = dailyTemp.Variables(0).ValuesAsNumpy()
daily_temperature_2m_min = dailyTemp.Variables(1).ValuesAsNumpy()

daily_temperature_data = {
	"date": pd.date_range(
		start = pd.to_datetime(dailyTemp.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(dailyTemp.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = dailyTemp.Interval()),
		inclusive = "left"
	).tz_convert(weatherResponse.Timezone().decode())
}

daily_temperature_data["temperature_2m_max"] = daily_temperature_2m_max
daily_temperature_data["temperature_2m_min"] = daily_temperature_2m_min

daily_temp_df = pd.DataFrame(data = daily_temperature_data)
print("\n This is the Daily Maximum and Minimum Temperature DataFrame:")
daily_temp_df.head(5)


 This is the Daily Maximum and Minimum Temperature DataFrame:


,date,temperature_2m_max,temperature_2m_min
0,2026-09-04 00:00:00+03:00,38.63,25.98
1,2026-09-05 00:00:00+03:00,38.08,26.83
2,2026-09-06 00:00:00+03:00,35.13,25.98
3,2026-09-07 00:00:00+03:00,36.18,26.43
4,2026-09-08 00:00:00+03:00,33.48,24.78


2. Manipulating Data for Asthma Exacerbation Risk calculations

According to studies one cannot rely solely on the current data values to predict the risk of a possible exacerbation. The diural temperature range (DTR) for example is more of an indicating factor than the raw current temperature value. The mean of the last 24 hours of air quality factors (nitrogen dioxide and PM2.5) and last 8 hours of ozone should be taken into account. As for the relative humidity the mean as well as the difference value of the last 24 hours have to be calculated. The difference value is by definition the current mean minus the previous mean of the same time period (24 hours). Generally, an asthma exacerbation depends mostly on the daily concentration of pollen with a lag of 0-4 days. 

In [10]:
hourly_air_quality_dataframe['Moving PM2.5 mean'] = hourly_air_quality_dataframe['pm2_5'].rolling(window = 24).mean()
hourly_air_quality_dataframe['Moving NO2 mean'] = hourly_air_quality_dataframe['nitrogen_dioxide'].rolling(window = 24).mean()
hourly_air_quality_dataframe['Moving O3 mean'] = hourly_air_quality_dataframe['ozone'].rolling(window = 8).mean()
hourly_air_quality_dataframe.head()

,date,pm2_5,nitrogen_dioxide,ozone,Moving PM2.5 mean,Moving NO2 mean,Moving O3 mean
0,2026-09-05 00:00:00+03:00,20.60,0.50,89.00,NaN,NaN,NaN
1,2026-09-05 01:00:00+03:00,19.00,0.50,85.00,NaN,NaN,NaN
2,2026-09-05 02:00:00+03:00,18.00,0.50,85.00,NaN,NaN,NaN
3,2026-09-05 03:00:00+03:00,16.90,0.50,86.00,NaN,NaN,NaN
4,2026-09-05 04:00:00+03:00,15.30,0.50,87.00,NaN,NaN,NaN


In [11]:
# Calculate the current PM2.5 mean over the last 24 hours
current_time = pd.Timestamp.now(hourly_air_quality_dataframe['date'].dt.tz)
print(current_time)
current_row = hourly_air_quality_dataframe[
    (hourly_air_quality_dataframe['date'].dt.date == current_time.date())&
    (hourly_air_quality_dataframe['date'].dt.hour == current_time.hour)
    ]
current_PM2_5_Mean = current_row['Moving PM2.5 mean'].iloc[0]
current_NO2_Mean = current_row['Moving NO2 mean'].iloc[0]
current_O3_Mean = current_row['Moving O3 mean'].iloc[0]
print(f"\n This is the current PM2.5 mean of the last 24 hours: {current_PM2_5_Mean:.2f}")
print(f"\n This is the current NO2 mean of the last 24 hours: {current_NO2_Mean:.2f}")
print(f"\n This is the current O3 mean of the last 8 hours: {current_O3_Mean:.2f}")

2026-09-06 11:35:34.648265+03:00

 This is the current PM2.5 mean of the last 24 hours: 13.95

 This is the current NO2 mean of the last 24 hours: 0.40

 This is the current O3 mean of the last 8 hours: 87.88


In [12]:
# Calculating:
# 1. the 72-hour mean for birch pollen,
hourly_pollen_df['Birch Pollen 72H mean'] = hourly_pollen_df['birch_pollen'].rolling(window = 72).mean()
# 2. 24-hour mean (daily) and shift for grass pollen,
hourly_pollen_df['Grass Pollen 24H mean'] = hourly_pollen_df['grass_pollen'].rolling(window = 24).mean()
hourly_pollen_df['Grass Pollen 72H Shift'] = hourly_pollen_df['Grass Pollen 24H mean'].shift(72)
# 3. and 72-hour mean for ragweed pollen:
hourly_pollen_df['Ragweed Pollen 72H mean'] = hourly_pollen_df['ragweed_pollen'].rolling(window = 72).mean()
hourly_pollen_df.iloc[96:110]

,date,birch_pollen,grass_pollen,ragweed_pollen,Birch Pollen 72H mean,Grass Pollen 24H mean,Grass Pollen 72H Shift,Ragweed Pollen 72H mean
96,2026-09-06 00:00:00+03:00,0.00,0.50,0.10,0.00,0.23,0.23,0.15
97,2026-09-06 01:00:00+03:00,0.00,0.40,0.10,0.00,0.22,0.23,0.14
98,2026-09-06 02:00:00+03:00,0.00,0.40,0.10,0.00,0.22,0.23,0.13
99,2026-09-06 03:00:00+03:00,0.00,0.30,0.10,0.00,0.21,0.22,0.13
100,2026-09-06 04:00:00+03:00,0.00,0.30,0.10,0.00,0.21,0.22,0.13
101,2026-09-06 05:00:00+03:00,0.00,0.20,0.10,0.00,0.21,0.21,0.12
102,2026-09-06 06:00:00+03:00,0.00,0.20,0.10,0.00,0.21,0.21,0.12
103,2026-09-06 07:00:00+03:00,0.00,0.20,0.10,0.00,0.21,0.20,0.12
104,2026-09-06 08:00:00+03:00,0.00,0.20,0.30,0.00,0.21,0.20,0.13
105,2026-09-06 09:00:00+03:00,0.00,0.20,0.20,0.00,0.21,0.20,0.13


In [13]:
current_time_Pollen = pd.Timestamp.now(hourly_pollen_df['date'].dt.tz)
print(f"\n Current local time: {current_time_Pollen}")
current_row_pollen = hourly_pollen_df[
    (hourly_pollen_df['date'].dt.date == current_time_Pollen.date())&
    (hourly_pollen_df['date'].dt.hour == current_time_Pollen.hour)
]
# Assigning Variable names for the required variables to be used in the risk assessment model. The variable names are self-explanatory.
Birch_Pollen_72H_mean = current_row_pollen['Birch Pollen 72H mean'].iloc[0]
print(f"\n This is the current Birch Pollen mean of the last 72 hours: {Birch_Pollen_72H_mean:.2f}")

Grass_Pollen_72H_lag = current_row_pollen['Grass Pollen 72H Shift'].iloc[0]
print(f"\n This is the mean Grass 24 Hour Pollen concentration with a lag of 3 days: {Grass_Pollen_72H_lag:.2f}")

Ragweed_Pollen_72H_mean = current_row_pollen['Ragweed Pollen 72H mean'].iloc[0]
print(f"\n This is the current Ragweed Pollen mean of the last 72 hours: {Ragweed_Pollen_72H_mean:.2f}")
current_row_pollen


 Current local time: 2026-09-06 11:35:35.493128+03:00

 This is the current Birch Pollen mean of the last 72 hours: 0.00

 This is the mean Grass 24 Hour Pollen concentration with a lag of 3 days: 0.20

 This is the current Ragweed Pollen mean of the last 72 hours: 0.13


,date,birch_pollen,grass_pollen,ragweed_pollen,Birch Pollen 72H mean,Grass Pollen 24H mean,Grass Pollen 72H Shift,Ragweed Pollen 72H mean
107,2026-09-06 11:00:00+03:00,0.00,0.20,0.10,0.00,0.21,0.20,0.13


In [14]:
hourly_RH_dataframe['Moving Relative Humidity mean'] = hourly_RH_dataframe['relative_humidity_2m'].rolling(window = 24).mean()
hourly_RH_dataframe['Shifted Moving Relative Humidity mean'] = hourly_RH_dataframe['Moving Relative Humidity mean'].shift(24)
print(f"\n DF showing mean and the preceding mean of Relative Humidity of 24 hours timeframe:")
hourly_RH_dataframe.loc[45:72]


 DF showing mean and the preceding mean of Relative Humidity of 24 hours timeframe:


,date,relative_humidity_2m,Moving Relative Humidity mean,Shifted Moving Relative Humidity mean
45,2026-09-05 21:00:00+03:00,28.00,36.54,NaN
46,2026-09-05 22:00:00+03:00,33.00,37.08,NaN
47,2026-09-05 23:00:00+03:00,39.00,37.54,23.42
48,2026-09-06 00:00:00+03:00,36.00,37.00,24.50
49,2026-09-06 01:00:00+03:00,39.00,36.46,25.62
50,2026-09-06 02:00:00+03:00,44.00,36.08,26.71
51,2026-09-06 03:00:00+03:00,47.00,35.83,27.75
52,2026-09-06 04:00:00+03:00,49.00,35.58,28.88
53,2026-09-06 05:00:00+03:00,50.00,35.42,29.92
54,2026-09-06 06:00:00+03:00,51.00,35.25,30.79


In [15]:
# Calculating current local time and the corresponding row in the hourly_RH_dataframe to get the current
# 24-hour mean of Relative Humidity and the previous 24-hour mean of Relative Humidity. Then, calculating
# the difference between the two means.
current_time_RH = pd.Timestamp.now(hourly_RH_dataframe['date'].dt.tz)
print(f"\n Current local time: {current_time_RH}")
current_row_RH = hourly_RH_dataframe[
    (hourly_RH_dataframe['date'].dt.date == current_time_RH.date())&
    (hourly_RH_dataframe['date'].dt.hour == current_time_RH.hour)
]
current_24_RH_Mean = current_row_RH['Moving Relative Humidity mean'].iloc[0]
prev_24_RH_Mean = current_row_RH['Shifted Moving Relative Humidity mean'].iloc[0]
print(f"\n This is the current 24-hour mean of Relative Humidity: {current_24_RH_Mean:.2f}")
print(f"\n This is the previous 24-hour mean of Relative Humidity: {prev_24_RH_Mean:.2f}")
mean_RH_difference = current_24_RH_Mean - prev_24_RH_Mean
print(f"\n The difference between the current 24-hour mean and the previous 24-hour mean of Relative Humidity is: {mean_RH_difference:.2f}")


 Current local time: 2026-09-06 11:35:36.095441+03:00

 This is the current 24-hour mean of Relative Humidity: 34.50

 This is the previous 24-hour mean of Relative Humidity: 33.75

 The difference between the current 24-hour mean and the previous 24-hour mean of Relative Humidity is: 0.75


In [16]:
daily_temp_df['DTR'] = daily_temp_df['temperature_2m_max'] - daily_temp_df['temperature_2m_min']
print(f"\n DF showing the daily maximum and minimum temperature and the difference between them:")
daily_temp_df.head()


 DF showing the daily maximum and minimum temperature and the difference between them:


,date,temperature_2m_max,temperature_2m_min,DTR
0,2026-09-04 00:00:00+03:00,38.63,25.98,12.65
1,2026-09-05 00:00:00+03:00,38.08,26.83,11.25
2,2026-09-06 00:00:00+03:00,35.13,25.98,9.15
3,2026-09-07 00:00:00+03:00,36.18,26.43,9.75
4,2026-09-08 00:00:00+03:00,33.48,24.78,8.70


In [17]:
current_date = pd.Timestamp.now(daily_temp_df['date'].dt.tz).date()
print(f"\n Current date: {current_date}")
current_row_temp = daily_temp_df[daily_temp_df['date'].dt.date == current_date]
current_temp_max = current_row_temp['temperature_2m_max'].iloc[0]
current_temp_min = current_row_temp['temperature_2m_min'].iloc[0]
current_temp_diff = current_temp_max - current_temp_min
print(f"\n Current maximum temperature: {current_temp_max:.2f}")
print(f"\n Current minimum temperature: {current_temp_min:.2f}")
print(f"\n Diurnal Temperature Range: {current_temp_diff:.2f}")


 Current date: 2026-09-06

 Current maximum temperature: 35.13

 Current minimum temperature: 25.98

 Diurnal Temperature Range: 9.15


3. Final Data Collection 

Collect all relevant Environmental Factors and their corresponding Values in one Dataset.

In [18]:
environmental_factors_df = pd.DataFrame({
    'Current PM2.5 Mean': [current_PM2_5_Mean],
    'Current NO2 Mean': [current_NO2_Mean],
    'Current O3 Mean': [current_O3_Mean],
    'Birch Pollen 72H Mean': [Birch_Pollen_72H_mean],
    'Grass Pollen 72H Lag': [Grass_Pollen_72H_lag],
    'Ragweed Pollen 72H Mean': [Ragweed_Pollen_72H_mean],
    'Mean RH Difference': [mean_RH_difference],
    'Diurnal Temp Range': [current_temp_diff]},
    index = [current_row_RH['date'].iloc[0]]
)
# environmental_factors_df = pd.DataFrame(data, index = ['current_date'])
environmental_factors_df

,Current PM2.5 Mean,Current NO2 Mean,Current O3 Mean,Birch Pollen 72H Mean,Grass Pollen 72H Lag,Ragweed Pollen 72H Mean,Mean RH Difference,Diurnal Temp Range
2026-09-06 11:00:00+03:00,13.95,0.40,87.88,0.00,0.20,0.13,0.75,9.15
